# LeetCode 3990 — Create Grid With Exactly K Paths II bằng Python

- Link: https://leetcode.com/problems/create-grid-with-exactly-k-paths-ii/
- Độ khó: **Hard**
- Chủ đề: Xây dựng (Construction), Bit, Tổ hợp, Đếm đường đi trên lưới (DP), Ma trận

> **Bài Premium: đề bài viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**
> Skeleton của bài Premium không có sẵn; tên phương thức dưới đây (`constructGrid`) là **dự đoán**, cần đối chiếu lại trên LeetCode (tên và kiểu trả về `List[str]`).
> Bài có **nhiều đáp án đúng**: LeetCode chấm bằng cách đếm số đường đi của lưới bạn trả về. Ví dụ trong đề chỉ là một trong các đáp án hợp lệ.

Skeleton (dự đoán, Python3):

```python
class Solution:
    def constructGrid(self, k: int) -> List[str]:
```

## 1. Giới thiệu & mục tiêu học

Cho số nguyên `k` (`1 ≤ k ≤ 1000`). Hãy **tự dựng** một lưới gồm ký tự `'.'` (ô trống) và `'#'` (chướng ngại), cỡ tối đa `25 × 25`, sao cho có **đúng `k`** đường đi từ ô trên-trái `(0, 0)` tới ô dưới-phải `(m-1, n-1)` chỉ đi **sang phải** hoặc **xuống**. Không có lưới nào thì trả mảng rỗng.

Đây là bài **xây dựng**, khác hẳn các bài tìm đáp án: ta được **chọn** `m`, `n` và hình dạng. Ý chính: số đường đi tới một ô là `f(i, j) = f(i-1, j) + f(i, j-1)` (nếu ô trống), nên ta có thể **thiết kế các giá trị `f`**. Ta dựng một "xương sống" nhân đôi giá trị ở mỗi bước (`1, 2, 4, 8, ...`) và hút từng luỹ thừa của 2 cần thiết (theo các bit của `k`) về đích qua các **làn** (lane) tách biệt.

Sau notebook này bạn sẽ:

1. Viết hàm **đếm đường đi** `count_paths(grid)` bằng DP — đây là "trọng tài" cho mọi lời giải.
2. Hiểu **tại sao** hình thoi `2 × 2` nhân đôi số đường đi, và cách xếp chuỗi hình thoi cho `2^t`.
3. Ghép các bit của `k` thành các làn riêng, kiểm chứng cho **mọi** `k` từ `1` đến `1000`.
4. Biết phép **nối tiếp** (series) hai lưới nhân số đường đi, dùng để thu nhỏ lưới khi `k` hợp số.
5. Hiểu vì sao test cho bài nhiều đáp án phải kiểm tra **tính chất** (đếm = `k`, kích thước ≤ 25), không so với một đáp án cố định.

## 2. Đề bài & ví dụ

**Phát biểu lại.**

- Dựng lưới `m × n` (`1 ≤ m, n ≤ 25`) chỉ gồm `'.'` và `'#'`, trả về dưới dạng danh sách `m` chuỗi dài `n`.
- Đường đi hợp lệ: dãy ô **trống** bắt đầu ở `(0, 0)`, kết thúc ở `(m-1, n-1)`, mỗi bước sang phải `(i, j+1)` hoặc xuống `(i+1, j)`.
- Trả về **bất kỳ** lưới nào có **đúng `k`** đường đi; nếu không tồn tại thì mảng rỗng.

**Ví dụ 1:** `k = 2` → `["..#", "#..", "#.."]`. Hai đường: `(0,0)→(0,1)→(1,1)→(1,2)→(2,2)` và `(0,0)→(0,1)→(1,1)→(2,1)→(2,2)`.

**Ví dụ 2:** `k = 3` → `["...", "#..", "#.."]` (ba đường đi).

**Ràng buộc**

- `1 <= k <= 1000`

Vì có nhiều đáp án, ta **không** assert `construct(k) == ví dụ`. Ta assert **`count_paths(construct(k)) == k`** và kích thước hợp lệ. Câu hỏi đáng nghĩ: với `1 ≤ k ≤ 1000` có luôn tồn tại lưới không? (Có — ta sẽ chứng minh bằng chính cách dựng.)

In [1]:
from typing import List

def count_paths(grid):
    """Số đường đi (phải/xuống) từ (0,0) tới (m-1,n-1) qua các ô '.'; DP f(i,j) = f(i-1,j) + f(i,j-1)."""
    m, n = len(grid), len(grid[0])
    if grid[0][0] == "#" or grid[m - 1][n - 1] == "#":
        return 0
    f = [[0] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            if grid[i][j] == "#":
                continue
            if i == 0 and j == 0:
                f[i][j] = 1
                continue
            up = f[i - 1][j] if i > 0 else 0
            left = f[i][j - 1] if j > 0 else 0
            f[i][j] = up + left
    return f[m - 1][n - 1]

def is_valid_shape(grid):
    return (1 <= len(grid) <= 25 and all(len(r) == len(grid[0]) for r in grid)
            and 1 <= len(grid[0]) <= 25 and all(ch in ".#" for r in grid for ch in r))

EXAMPLES = [(2, ["..#", "#..", "#.."]), (3, ["...", "#..", "#.."])]
for k, g in EXAMPLES:
    print(k, g, "-> đếm được", count_paths(g), "hợp lệ kích thước:", is_valid_shape(g))
    assert count_paths(g) == k

2 ['..#', '#..', '#..'] -> đếm được 2 hợp lệ kích thước: True
3 ['...', '#..', '#..'] -> đếm được 3 hợp lệ kích thước: True


## 3. Kiến thức nền

### 3.1. Đếm đường đi trên lưới bằng DP

Gọi `f(i, j)` là số đường đi từ `(0, 0)` tới `(i, j)`. Ô chướng ngại: `f = 0`. Ô trống: `f(0, 0) = 1`, còn lại `f(i, j) = f(i-1, j) + f(i, j-1)` (ô trên cộng ô trái, thiếu ô thì coi `0`). Đây là điều ta sẽ "điều khiển" khi thiết kế lưới.

In [2]:
def f_table(grid):
    m, n = len(grid), len(grid[0])
    f = [[0] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            if grid[i][j] == "#":
                continue
            if i == 0 and j == 0:
                f[i][j] = 1
            else:
                f[i][j] = (f[i - 1][j] if i else 0) + (f[i][j - 1] if j else 0)
    return f

for row in f_table(["...", "...", "..."]):
    print(row)          # lưới trống 3x3: các số tổ hợp C(i+j, i), tới đích là 6

[1, 1, 1]
[1, 2, 3]
[1, 3, 6]


### 3.2. Thao tác bit: `k >> t & 1`, `bit_length()`

- `k >> t & 1` lấy bit thứ `t` của `k` (bit `0` là bit thấp nhất): `1` nếu `2^t` nằm trong biểu diễn nhị phân.
- `k.bit_length()` là số bit cần để biểu diễn `k` (với `k ≥ 1`, bit cao nhất là bit `bit_length() - 1` và luôn bằng `1`).
- Mọi `k` bằng tổng các `2^t` ứng với bit bật: `k = Σ bit_t · 2^t`.

In [3]:
k = 1000
print(bin(k), k.bit_length())
print([t for t in range(k.bit_length()) if k >> t & 1])        # các bit bật
print(sum(1 << t for t in range(k.bit_length()) if k >> t & 1) == k)

0b1111101000 10
[3, 5, 6, 7, 8, 9]
True


### 3.3. Tạo lưới bằng list-of-lists rồi ghép thành chuỗi

Lưới ban đầu đặt toàn `'#'`; ta "mở" từng ô bằng `g[i][j] = '.'`; cuối cùng `["".join(row) for row in g]`. Lưu ý **không** dùng `[['#'] * n] * m` (các hàng cùng tham chiếu một list!).

In [4]:
bad = [["#"] * 3] * 2
bad[0][0] = "."
print(bad)                                           # cả hai hàng cùng đổi!
good = [["#"] * 3 for _ in range(2)]
good[0][0] = "."
print(good, ["".join(r) for r in good])

[['.', '#', '#'], ['.', '#', '#']]
[['.', '#', '#'], ['#', '#', '#']] ['.##', '###']


## 4. Ý tưởng nền tảng / trực giác

Hãy thử vài lưới nhỏ bằng tay:

- Một hàng `1 × n` toàn `'.'`: `1` đường. Một lưới `m × n` trống: `C(m+n-2, m-1)` đường (`3×3` có `6`).
- **Hình thoi:** các ô `(0,0), (0,1), (1,0), (1,1)`: từ `(0,0)` tới `(1,1)` có **2** đường (phải-xuống, xuống-phải). Đó là cách **nhân đôi**.

Nếu ta nối các hình thoi: `(1,1)` trở thành điểm đầu của hình thoi kế, ta nhân đôi tiếp: `1 → 2 → 4 → 8 → ...`. Chuỗi hình thoi cho đường đi `2^t`.

Để được `k` bất kỳ ta cần các luỹ thừa 2 **khác nhau** cộng lại: `k = 2^{t1} + 2^{t2} + ...`. Mỗi số hạng `2^t` có thể lấy ra ở **giai đoạn `t`** của chuỗi nhân đôi và dẫn về đích riêng, rồi các luồng này cộng lại ở đích. Muốn cộng được, các nhánh **không được dính vào nhau** trên đường về, vì nếu dính thì số đường đi sẽ lan sang chỗ không mong muốn.

Ta thử trực tiếp: dựng chuỗi hình thoi và in bảng `f`.

In [5]:
def diamond_chain(T):
    """Lưới (2T-1)x(2T-1)... chỉ gồm dải |i-j| <= 1: số đường tới (i,i) là 2^i."""
    n = T + 1
    g = [["#"] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            if abs(i - j) <= 1:
                g[i][j] = "."
    return ["".join(r) for r in g]

g = diamond_chain(4)
for row in g:
    print(row)
for row in f_table(g):
    print(row)
print("đường đi tới đích (4,4):", count_paths(g), " = 2^4 =", 2 ** 4)

..###
...##
#...#
##...
###..
[1, 1, 0, 0, 0]
[1, 2, 2, 0, 0]
[0, 2, 4, 4, 0]
[0, 0, 4, 8, 8]
[0, 0, 0, 8, 16]
đường đi tới đích (4,4): 16  = 2^4 = 16


## 5. Cách 1 — Brute Force: liệt kê mọi lưới nhỏ

**Ý tưởng.** Với một kích thước `(m, n)` nhỏ, thử **mọi** lưới (mỗi ô `.` hoặc `#`, hai ô góc phải trống) rồi đếm đường đi bằng DP. Ghi lại với mỗi `k` lưới có **diện tích nhỏ nhất**. Nó chỉ chạy được khi `m·n ≤ 16` (`2^14` lưới), nhưng cho ta:

- một trọng tài đúng tuyệt đối cho `k` nhỏ (giá trị lớn nhất của lưới `4×4` trống là `20`);
- cảm nhận rằng "hầu như mọi `k` nhỏ đều có lưới nhỏ".

In [6]:
import itertools

def best_small_grids(max_cells=16):
    best = {}                                   # k -> (diện tích, lưới)
    for m in range(1, 5):
        for n in range(1, 5):
            if m * n > max_cells or m * n < 1:
                continue
            cells = [(i, j) for i in range(m) for j in range(n)]
            fixed = {(0, 0), (m - 1, n - 1)}
            free_cells = [c for c in cells if c not in fixed]
            for mask in range(1 << len(free_cells)):
                g = [["#"] * n for _ in range(m)]
                for (i, j) in fixed:
                    g[i][j] = "."
                for idx, (i, j) in enumerate(free_cells):
                    if mask >> idx & 1:
                        g[i][j] = "."
                rows = ["".join(r) for r in g]
                c = count_paths(rows)
                if c >= 1 and (c not in best or m * n < best[c][0]):
                    best[c] = (m * n, rows)
    return best

small = best_small_grids()
print("các k đạt được trong lưới ≤ 16 ô:", sorted(small))
for k in (1, 2, 3, 4, 5, 6, 7, 8):
    area, rows = small[k]
    print(k, "diện tích", area, rows)

các k đạt được trong lưới ≤ 16 ô: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 18, 19, 20]
1 diện tích 1 ['.']
2 diện tích 4 ['..', '..']
3 diện tích 6 ['...', '...']
4 diện tích 8 ['....', '....']
5 diện tích 9 ['...', '...', '#..']
6 diện tích 9 ['...', '...', '...']
7 diện tích 12 ['....', '....', '##..']
8 diện tích 12 ['...#', '....', '#...']


**Độ phức tạp.** `O(2^{mn} · mn)` cho mỗi kích thước — chỉ khả thi với lưới tí hon; với `25 × 25` là bất khả thi. Nhưng ta rút ra hai điều: (a) trong lưới `≤ 16` ô không phải mọi `k` đều đạt được (ví dụ `k = 17` không có trong danh sách in ra ở trên, dù `18, 19, 20` có); (b) cần một **quy tắc dựng** thay vì tìm kiếm. 

## 6. Bước đột phá — nhân đôi + làn nhị phân

Ta xây xương sống gồm các "tầng" `t = 0, 1, ..., T−1` với `T = k.bit_length()`. Đặt `S_t = (2t, 2t)` là ô đầu tầng `t`. Mỗi tầng gồm 4 ô hình thoi và 1 ô nối sang tầng sau:

- `(2t, 2t+1)` (góc trên-phải) và `(2t+1, 2t)` (góc dưới-trái): mỗi ô nhận `f = f(S_t)`.
- `(2t+1, 2t+1)` (đáy): nhận từ cả hai ô trên ⇒ `f = 2·f(S_t)`.
- `(2t+1, 2t+2)` (ô nối): nhận từ ô đáy bên trái ⇒ `f = 2·f(S_t)`.
- `S_{t+1} = (2t+2, 2t+2)`: nhận từ ô nối ở trên ⇒ `f(S_{t+1}) = 2·f(S_t)`.

Vậy `f(S_t) = 2^t`, và ô dưới-trái `(2t+1, 2t)` có `f = 2^t`.

**Làn nhị phân.** Từ ô dưới-trái `(2t+1, 2t)` ta kéo một **làn đi thẳng xuống** dọc cột `2t` tới hàng cuối `m-1 = 2T`. Cột chẵn `2t` chỉ mang giá trị `2^t`. Hàng cuối được mở **toàn bộ** để mọi làn đổ về bên phải; đích `(m-1, n-1)` ở cuối hàng: `f(đích) = Σ f(đáy làn) = Σ_{bit t bật} 2^t = k`.

**Quy tắc "nếu … thì …":**

- nếu bit `t` của `k` bật thì mở làn `t` (các ô cột `2t`, hàng `2t+1 .. 2T`);
- nếu bit `t` tắt thì **chặn** làn (ô `(2t+2, 2t)` vẫn là `'#'`), ô dưới-trái là ngõ cụt;
- cột lẻ giữa các làn luôn là `'#'` (ngoài hàng cuối và các ô hình thoi) nên các làn **không chạm nhau**.

**Kích thước:** `m = 2T + 1`, `n = 2T − 1`. Với `k ≤ 1000` ta có `T ≤ 10`, nên tối đa `21 × 19` — vừa trong `25 × 25`.

## 7. Các cách tiếp theo

- **Cách 2:** xương sống hình thoi + làn nhị phân — đáp án nộp.
- **Cách 3:** **nối tiếp** hai lưới (nhân số đường đi), dùng để chia `k = a · b` và thu nhỏ lưới.

### 7.1. Cách 2 — Xương sống hình thoi + làn nhị phân

**Giải thích từng đoạn code.**

1. `T = k.bit_length()`, `m = 2T+1`, `n = 2T-1`; tạo lưới toàn `'#'`.
2. Với mỗi tầng `t`: mở `S_t`; nếu còn chỗ (`2t+1 < n`) mở ô trên-phải và đáy; luôn mở ô dưới-trái; nếu `t < T-1` mở ô nối sang tầng kế.
3. Với mỗi bit `t` bật: mở các ô `(r, 2t)` với `r = 2t+1 .. m-1` (làn đi xuống).
4. Mở toàn bộ hàng cuối.
5. Chuyển thành danh sách chuỗi.

Tầng cao nhất (`t = T-1`) không cần hình thoi đầy đủ vì ta chỉ cần ô dưới-trái của nó.

In [7]:
def construct_binary_lanes(k):
    T = k.bit_length()
    m, n = 2 * T + 1, 2 * T - 1
    g = [["#"] * n for _ in range(m)]
    for t in range(T):
        g[2 * t][2 * t] = "."                      # S_t
        g[2 * t + 1][2 * t] = "."                  # góc dưới-trái
        if 2 * t + 1 < n:
            g[2 * t][2 * t + 1] = "."              # góc trên-phải
            g[2 * t + 1][2 * t + 1] = "."          # đáy
        if t < T - 1:
            g[2 * t + 1][2 * t + 2] = "."          # ô nối sang tầng t+1
    for t in range(T):
        if k >> t & 1:
            for r in range(2 * t + 1, m):          # làn t chạy xuống tận hàng cuối
                g[r][2 * t] = "."
    for j in range(n):                             # hàng cuối đổ về đích
        g[m - 1][j] = "."
    return ["".join(row) for row in g]

for k in (1, 2, 3, 6, 11):
    g = construct_binary_lanes(k)
    print(f"k={k}: {len(g)}x{len(g[0])}, đếm = {count_paths(g)}")
    assert count_paths(g) == k
print()
for row in construct_binary_lanes(6):
    print(row)

k=1: 3x1, đếm = 1
k=2: 5x3, đếm = 2
k=3: 5x3, đếm = 3
k=6: 7x5, đếm = 6
k=11: 9x7, đếm = 11

..###
...##
##..#
##...
##.#.
##.#.
.....


**Độ phức tạp.** Thời gian và bộ nhớ `O(m·n) = O(log² k)` (lưới có `O(log k)` hàng/cột). Kích thước tối đa `21 × 19` khi `k ≤ 1000`.

### 7.2. Cách 3 — Nối tiếp hai lưới (nhân)

**Ý tưởng.** Cho lưới `A` (`m1 × n1`, có `a` đường) và `B` (`m2 × n2`, có `b` đường). Đặt `A` ở góc trên-trái và `B` ở góc dưới-phải **chung ô góc** `(m1-1, n1-1) = B(0,0)`; mọi ô còn lại là `'#'`. Kích thước `(m1+m2-1) × (n1+n2-1)`.

Mọi đường đi phải đi qua ô chung (từ khối `A` không thể bước thẳng sang khối `B` nếu không qua ô góc: bước sang phải từ cột cuối của `A` ở hàng `< m1-1` rơi vào ô trống `'#'`, bước xuống từ hàng cuối ở cột `< n1-1` cũng vậy). Vậy số đường đi = `a · b`.

Dùng cho hợp số: `k = a·b` → `series(build(a), build(b))`. Với số nguyên tố ta rơi về cách 2. Hai viên gạch cơ bản: lưới `["."]` (1 đường, kích thước `1×1`) và hình thoi `3×3` ví dụ 1 (`["..#", "#..", "#.."]`, 2 đường).

In [8]:
from functools import lru_cache

def series(A, B):
    m1, n1, m2, n2 = len(A), len(A[0]), len(B), len(B[0])
    m, n = m1 + m2 - 1, n1 + n2 - 1
    g = [["#"] * n for _ in range(m)]
    for i in range(m1):
        for j in range(n1):
            if A[i][j] == ".":
                g[i][j] = "."
    for i in range(m2):
        for j in range(n2):
            if B[i][j] == ".":
                g[m1 - 1 + i][n1 - 1 + j] = "."
    return ["".join(r) for r in g]

DIAMOND = ["..#", "#..", "#.."]
A, B = construct_binary_lanes(3), DIAMOND
C = series(A, B)
print(len(A), len(A[0]), "x", len(B), len(B[0]), "->", len(C), len(C[0]))
print(count_paths(A), count_paths(B), count_paths(C), "(tích:", count_paths(A) * count_paths(B), ")")
assert count_paths(C) == count_paths(A) * count_paths(B)

@lru_cache(None)
def construct_compose(k):
    """Chọn lưới nhỏ nhất theo diện tích giữa: cách 2, hoặc tích a*b của hai lưới đã dựng."""
    if k == 1:
        return (".",)
    if k == 2:
        return tuple(DIAMOND)
    best = tuple(construct_binary_lanes(k))
    for a in range(2, int(k ** 0.5) + 1):
        if k % a == 0:
            g = tuple(series(construct_compose(a), construct_compose(k // a)))
            if len(g) * len(g[0]) < len(best) * len(best[0]) and len(g) <= 25 and len(g[0]) <= 25:
                best = g
    return best

for k in (1, 2, 6, 12, 36, 64, 100, 720, 997):
    g = construct_compose(k)
    print(f"k={k}: {len(g)}x{len(g[0])} (cách 2: {len(construct_binary_lanes(k))}x{len(construct_binary_lanes(k)[0])}), đếm = {count_paths(list(g))}")
    assert count_paths(list(g)) == k

5 3 x 3 3 -> 7 5
3 2 6 (tích: 6 )
k=1: 1x1 (cách 2: 3x1), đếm = 1
k=2: 3x3 (cách 2: 5x3), đếm = 2
k=6: 7x5 (cách 2: 7x5), đếm = 6
k=12: 9x7 (cách 2: 9x7), đếm = 12
k=36: 13x9 (cách 2: 13x11), đếm = 36
k=64: 13x13 (cách 2: 15x13), đếm = 64
k=100: 15x13 (cách 2: 15x13), đếm = 100
k=720: 21x17 (cách 2: 21x19), đếm = 720
k=997: 21x19 (cách 2: 21x19), đếm = 997


**Độ phức tạp.** Thời gian dựng `O(d(k) · (kích thước)²)` với `d(k)` là số ước. Phép nối tiếp giữ nguyên tính đúng (nhân), nhưng kích thước cộng nên không phải lúc nào cũng nhỏ hơn cách 2; ta chọn cái nhỏ nhất.

## 8. TRACE từng bước

Ta in lưới và bảng `f` cho `k = 11 = 1011₂` (`T = 4`). Quan sát: các giá trị trên xương sống là `1, 2, 4, 8`; làn 0, 1, 3 (bit 0, 1, 3 bật) mang `1, 2, 8` về hàng cuối; làn 2 (bit tắt) bị chặn.

| ô | ý nghĩa |
|---|---|
| `(2t, 2t)` | `f = 2^t` (xương sống) |
| `(2t+1, 2t)` | góc dưới-trái, `f = 2^t`, đầu làn `t` |
| hàng cuối | cộng dồn các làn từ trái sang phải |

In [9]:
def trace_construct(k):
    T = k.bit_length()
    print(f"k = {k} = {bin(k)}, T = {T}, kích thước {2 * T + 1} x {2 * T - 1}; bit bật:", [t for t in range(T) if k >> t & 1])
    g = construct_binary_lanes(k)
    f = f_table(g)
    print("lưới:")
    for row in g:
        print("  ", row)
    print("bảng f (đường đi tới từng ô):")
    for row in f:
        print("  ", " ".join(f"{v:3d}" for v in row))
    print("xương sống f(S_t):", [f[2 * t][2 * t] for t in range(T)])
    print("hàng cuối:", f[-1])
    print("đích =", f[-1][-1])
    return f[-1][-1]

trace_construct(11)

k = 11 = 0b1011, T = 4, kích thước 9 x 7; bit bật: [0, 1, 3]
lưới:
   ..#####
   ...####
   .#..###
   .#...##
   .#.#..#
   .#.#...
   .#.###.
   .#.###.
   .......
bảng f (đường đi tới từng ô):
     1   1   0   0   0   0   0
     1   2   2   0   0   0   0
     1   0   2   2   0   0   0
     1   0   2   4   4   0   0
     1   0   2   0   4   4   0
     1   0   2   0   4   8   8
     1   0   2   0   0   0   8
     1   0   2   0   0   0   8
     1   1   3   3   3   3  11
xương sống f(S_t): [1, 2, 4, 8]
hàng cuối: [1, 1, 3, 3, 3, 3, 11]
đích = 11


11

Hàng cuối của `k = 11` in ra `[1, 1, 3, 3, 3, 3, 11]`: làn 0 (cột 0) đổ `1`, làn 1 (cột 2) đổ `2` (cộng dồn `1 + 2 = 3`), làn 2 (cột 4) bị chặn nên chỉ chuyển tiếp, làn 3 (cột 6) đổ `8` ⇒ `3 + 8 = 11`. Ta trace thêm `k = 8` (chỉ một bit bật, nên chỉ làn cao nhất):

In [10]:
trace_construct(8)

k = 8 = 0b1000, T = 4, kích thước 9 x 7; bit bật: [3]
lưới:
   ..#####
   ...####
   ##..###
   ##...##
   ####..#
   ####...
   ######.
   ######.
   .......
bảng f (đường đi tới từng ô):
     1   1   0   0   0   0   0
     1   2   2   0   0   0   0
     0   0   2   2   0   0   0
     0   0   2   4   4   0   0
     0   0   0   0   4   4   0
     0   0   0   0   4   8   8
     0   0   0   0   0   0   8
     0   0   0   0   0   0   8
     0   0   0   0   0   0   8
xương sống f(S_t): [1, 2, 4, 8]
hàng cuối: [0, 0, 0, 0, 0, 0, 8]
đích = 8


8

## 9. Code nộp LeetCode

Tên phương thức dự đoán `constructGrid` (cần đối chiếu lại trên LeetCode).

In [11]:
from typing import List

class Solution:
    def constructGrid(self, k: int) -> List[str]:
        T = k.bit_length()
        m, n = 2 * T + 1, 2 * T - 1
        g = [["#"] * n for _ in range(m)]
        for t in range(T):
            g[2 * t][2 * t] = "."                  # đầu tầng t: f = 2^t
            g[2 * t + 1][2 * t] = "."              # góc dưới-trái: f = 2^t
            if 2 * t + 1 < n:
                g[2 * t][2 * t + 1] = "."          # góc trên-phải
                g[2 * t + 1][2 * t + 1] = "."      # đáy: f = 2^(t+1)
            if t < T - 1:
                g[2 * t + 1][2 * t + 2] = "."      # ô nối sang tầng t+1
        for t in range(T):
            if k >> t & 1:                         # làn t chạy xuống hàng cuối
                for r in range(2 * t + 1, m):
                    g[r][2 * t] = "."
        for j in range(n):                         # hàng cuối gom các làn về đích
            g[m - 1][j] = "."
        return ["".join(row) for row in g]

for k, ex in EXAMPLES:
    g = Solution().constructGrid(k)
    assert is_valid_shape(g) and count_paths(g) == k
print("Code nộp: các ví dụ cho đúng số đường đi (đáp án khác ví dụ nhưng hợp lệ)")
print(Solution().constructGrid(2))

Code nộp: các ví dụ cho đúng số đường đi (đáp án khác ví dụ nhưng hợp lệ)
['..#', '...', '##.', '##.', '...']


## 10. Vì sao thuật toán đúng

Gọi `f(i, j)` là số đường đi tới `(i, j)`. Ta kiểm tra lần lượt theo hàng, mỗi bước chỉ có hai nguồn: ô trên và ô trái.

**(a) Xương sống.** `f(0,0) = 1`. Với tầng `t`, giả sử `f(S_t) = 2^t` với `S_t = (2t, 2t)`:

- Ô `(2t, 2t+1)`: ô trên là `(2t−1, 2t+1)`. Ta chỉ mở ở hàng lẻ các ô cột `2t'` (làn/dưới-trái), cột `2t'+1` (đáy), cột `2t'+2` (ô nối). Hàng `2t−1` có ô nối của tầng `t−1` ở cột `2t`, đáy ở cột `2t−1`, góc dưới-trái ở cột `2t−2`; **không** có ô nào ở cột `2t+1`. Nên ô trên chặn; ô trái là `S_t`. Vậy `f = 2^t`.
- Ô `(2t+1, 2t)`: ô trên là `S_t`; ô trái `(2t+1, 2t−1)` (cột lẻ, hàng lẻ) là `'#'` (chỉ đáy tầng `t'` ở `(2t'+1, 2t'+1)`, tức cột chẵn khác `2t-1` — kiểm tra `2t'+1 = 2t−1` ⇒ hàng `2t'+1 = 2t−1 ≠ 2t+1`). Vậy `f = 2^t`.
- Ô đáy `(2t+1, 2t+1)`: ô trên `(2t, 2t+1)` và ô trái `(2t+1, 2t)` đều `2^t` ⇒ `f = 2^{t+1}`.
- Ô nối `(2t+1, 2t+2)`: ô trên `(2t, 2t+2)` là `'#'` (ở hàng `2t` chỉ mở `S_t` và ô trên-phải), ô trái là đáy ⇒ `f = 2^{t+1}`.
- `S_{t+1} = (2t+2, 2t+2)`: ô trên là ô nối `2^{t+1}`; ô trái `(2t+2, 2t+1)` (hàng chẵn, cột lẻ) là `'#'` (hàng chẵn `2t+2` chỉ có `S_{t+1}`, ô trên-phải `(2t+2, 2t+3)`, và các ô làn cột chẵn) ⇒ `f = 2^{t+1}`. ✓

**(b) Làn.** Ô `(r, 2t)` với `r ≥ 2t+2` nằm trong cột chẵn; ô trái là cột lẻ `2t−1`: ở các hàng `r ≥ 2t+2` cột `2t−1` chỉ có thể mở nếu `r = 2t−2` hoặc `r = 2t−1` (ô trên-phải và đáy của tầng `t−1`), vì `r ≥ 2t+2` nên **không**. Còn ngoài hàng cuối, mọi ô cột lẻ khác là `'#'`. Vậy ô làn chỉ nhận từ ô trên ⇒ giữ nguyên `f = 2^t` xuống tới hàng `m−2`. Hàng cuối: ô `(m−1, 2t)` nhận ô trên (`2^t` nếu bit `t` bật, `0` nếu làn bị chặn) cộng ô trái.

**(c) Hàng cuối.** Ô `(m−1, j)` có ô trên: chỉ cột làn (`j = 2t`, bit bật) mới có ô trên mở; cột lẻ có ô trên `(m−2, j)` là `'#'` (hàng `m−2 = 2T−1` chỉ mở ô dưới-trái của tầng `T−1` ở cột `2T−2 = n−1`). Vậy `f(m−1, j) = f(m−1, j−1) + [bit t bật]·2^t` (với `j = 2t`) ⇒ cộng dồn các làn. Ô đích `(m−1, n−1)`, `n−1 = 2T−2` là đáy của làn `T−1` (bit cao nhất luôn bật), vậy `f(đích) = Σ_{bit t bật} 2^t = k`. ∎

**(d) Kích thước.** `T = bit_length(k) ≤ 10` khi `k ≤ 1000 < 1024` ⇒ `m = 2T+1 ≤ 21`, `n = 2T−1 ≤ 19`. Luôn có lưới nên không bao giờ trả mảng rỗng.

Phần dưới ta **kiểm chứng toàn bộ** `k = 1..1000` bằng `count_paths` để không phụ thuộc vào lập luận tay.

## 11. Phân tích Time/Space

Gọi `T = ⌈log₂(k+1)⌉ = k.bit_length()`.

| Cách | Time | Space | Kích thước lưới |
|---|---|---|---|
| Liệt kê lưới nhỏ | `O(2^{mn} · mn)` | `O(mn)` | tối thiểu theo diện tích (chỉ `k` nhỏ) |
| Hình thoi + làn nhị phân (nộp) | `O(T²)` | `O(T²)` | `(2T+1) × (2T−1)`, tối đa `21 × 19` |
| Nối tiếp theo ước | `O(d(k) · T²)` | `O(T²)` | nhỏ hơn hoặc bằng cách trên (chọn min) |

Kiểm tra kích thước lớn nhất và thời gian cho cả 1000 giá trị:

In [12]:
import time
t0 = time.time()
max_m = max_n = 0
for k in range(1, 1001):
    g = Solution().constructGrid(k)
    max_m = max(max_m, len(g)); max_n = max(max_n, len(g[0]))
print("kích thước tối đa trên k=1..1000:", max_m, "x", max_n, "; thời gian dựng + chưa đếm:", round(time.time() - t0, 3), "giây")
print("với T=12 (k < 4096) ta cần", 2 * 12 + 1, "x", 2 * 12 - 1, "-> vẫn ≤ 25")

kích thước tối đa trên k=1..1000: 21 x 19 ; thời gian dựng + chưa đếm: 0.007 giây
với T=12 (k < 4096) ta cần 25 x 23 -> vẫn ≤ 25


## 12. Bộ test

Bài có **nhiều đáp án**, nên test theo **tính chất**: lưới hợp lệ (≤ 25×25, chỉ `.`/`#`, hai góc trống) và `count_paths == k`. Ta kiểm chứng `count_paths` bằng một bộ đếm khác (liệt kê đường bằng DFS) trên lưới nhỏ.

In [13]:
S = Solution()
# 1) Ví dụ đề (kiểm tra tính chất, không so khớp chuỗi)
for k, ex in EXAMPLES:
    g = S.constructGrid(k)
    assert is_valid_shape(g) and g[0][0] == "." and g[-1][-1] == "." and count_paths(g) == k
print("Ví dụ đề: OK")

# 2) Toàn bộ k = 1..1000
for k in range(1, 1001):
    g = S.constructGrid(k)
    assert is_valid_shape(g), k
    assert g[0][0] == "." and g[-1][-1] == ".", k
    assert count_paths(g) == k, (k, count_paths(g))
print("k = 1..1000: mọi lưới hợp lệ và đếm đúng k")

Ví dụ đề: OK
k = 1..1000: mọi lưới hợp lệ và đếm đúng k


In [14]:
# 3) Trọng tài: bộ đếm bằng DFS liệt kê từng đường (khác DP), kiểm tra trên lưới nhỏ ngẫu nhiên và lưới cách 1
import random, sys
sys.setrecursionlimit(10000)

def count_paths_dfs(grid):
    m, n = len(grid), len(grid[0])
    if grid[0][0] == "#" or grid[m - 1][n - 1] == "#":
        return 0
    def go(i, j):
        if i == m - 1 and j == n - 1:
            return 1
        total = 0
        if j + 1 < n and grid[i][j + 1] == ".":
            total += go(i, j + 1)
        if i + 1 < m and grid[i + 1][j] == ".":
            total += go(i + 1, j)
        return total
    return go(0, 0)

rng = random.Random(2026)
for _ in range(600):
    m, n = rng.randint(1, 6), rng.randint(1, 6)
    g = ["".join(rng.choice(".#.") for _ in range(n)) for _ in range(m)]
    assert count_paths(g) == count_paths_dfs(g), g
for k in range(1, 13):                       # lưới của cách dựng với k nhỏ: đếm DFS cũng ra k
    assert count_paths_dfs(S.constructGrid(k)) == k
print("DP ↔ DFS khớp trên 600 lưới ngẫu nhiên và các lưới dựng cho k ≤ 12")

# 4) Cách 1: kết quả liệt kê trùng với bộ đếm
for k, (area, rows) in small.items():
    assert count_paths(rows) == k == count_paths_dfs(rows)
print("Lưới tối thiểu từ liệt kê: khớp bộ đếm")

DP ↔ DFS khớp trên 600 lưới ngẫu nhiên và các lưới dựng cho k ≤ 12
Lưới tối thiểu từ liệt kê: khớp bộ đếm


In [15]:
# 5) Cách 3 (nối tiếp) cho mọi k: đúng số đường và kích thước ≤ 25
sizes_better = 0
for k in range(1, 1001):
    g = list(construct_compose(k))
    assert is_valid_shape(g) and count_paths(g) == k, k
    g2 = S.constructGrid(k)
    if len(g) * len(g[0]) < len(g2) * len(g2[0]):
        sizes_better += 1
print("nối tiếp: mọi k đúng; nhỏ hơn cách 2 ở", sizes_better, "giá trị k")

# 6) Phép nối tiếp nhân số đường đi trên lưới ngẫu nhiên nhỏ
for _ in range(200):
    def rnd():
        m, n = rng.randint(1, 4), rng.randint(1, 4)
        g = [[rng.choice(".#.") for _ in range(n)] for _ in range(m)]
        g[0][0] = g[-1][-1] = "."
        return ["".join(r) for r in g]
    A, B = rnd(), rnd()
    assert count_paths(series(A, B)) == count_paths(A) * count_paths(B), (A, B)
print("series: count(A⊕B) = count(A)·count(B) trên 200 cặp ngẫu nhiên")

nối tiếp: mọi k đúng; nhỏ hơn cách 2 ở 404 giá trị k
series: count(A⊕B) = count(A)·count(B) trên 200 cặp ngẫu nhiên


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Các làn đặt sát nhau (không có cột trống giữa chúng)

Nếu làn `t` đặt ở cột `t` thay vì `2t`, các làn liền kề: ô `(r, t)` nhận thêm từ ô trái `(r, t-1)`, nên giá trị chảy ngang giữa các làn và đếm bị sai.

In [16]:
def wrong_adjacent_lanes(k):
    T = k.bit_length()
    m, n = 2 * T + 1, 2 * T - 1
    g = [["#"] * n for _ in range(m)]
    for t in range(T):
        g[2 * t][2 * t] = "."
        g[2 * t + 1][2 * t] = "."
        if 2 * t + 1 < n:
            g[2 * t][2 * t + 1] = "."
            g[2 * t + 1][2 * t + 1] = "."
        if t < T - 1:
            g[2 * t + 1][2 * t + 2] = "."
    for t in range(T):
        if k >> t & 1:
            for r in range(2 * t + 1, m):
                g[r][2 * t] = "."
                if 2 * t + 1 < n:
                    g[r][2 * t + 1] = "."          # SAI: làn dày 2 cột, sát cột kế
    for j in range(n):
        g[m - 1][j] = "."
    return ["".join(r) for r in g]

bad = [k for k in range(1, 200) if count_paths(wrong_adjacent_lanes(k)) != k]
print("số k bị sai trong 1..199:", len(bad), "; ví dụ:", bad[:8])
k = bad[0]
print("k =", k, "đếm sai =", count_paths(wrong_adjacent_lanes(k)))

số k bị sai trong 1..199: 191 ; ví dụ: [3, 5, 6, 7, 9, 10, 11, 12]
k = 3 đếm sai = 14


**Cách đúng:** mỗi làn **một ô rộng**, hai làn cách nhau một cột `'#'`.

### Lỗi 2 — Dựng kiểu "mỗi đường một cột" (tuyến tính) — vượt quá 25

Ý tưởng ngây thơ: lưới 2 hàng, dùng "số cột chồng nhau" làm số đường. Cách này cho `k` đường với chiều rộng khoảng `k` — với `k = 1000` là `1000 > 25`.

In [17]:
def naive_two_rows(k):
    # hàng 0: k+1 ô trống từ cột 0 tới cột k, hàng 1: k+1 ô trống từ cột 0 tới cột k
    # mỗi vị trí 'chuyển hàng' cho một đường -> khoảng k+1 đường (đủ để minh hoạ)
    n = k
    return ["." * n, "." * n]

for k in (3, 10, 30):
    g = naive_two_rows(k)
    print(k, "kích thước", len(g), "x", len(g[0]), "đếm =", count_paths(g), "hợp lệ kích thước:", is_valid_shape(g))

3 kích thước 2 x 3 đếm = 3 hợp lệ kích thước: True
10 kích thước 2 x 10 đếm = 10 hợp lệ kích thước: True
30 kích thước 2 x 30 đếm = 30 hợp lệ kích thước: False


**Vì sao sai:** đường đi tăng tuyến tính theo chiều rộng nên vượt giới hạn `25` rất sớm. **Cách đúng:** dùng cấu trúc cho số đường đi tăng **theo luỹ thừa** (nhân đôi mỗi tầng) → chỉ `O(log k)` kích thước.

### Lỗi 3 — Quên chặn làn của bit tắt (mở mọi làn)

Mở mọi làn cho ra tổng `2^T − 1` (mọi bit bật).

In [18]:
def wrong_all_lanes(k):
    T = k.bit_length()
    m, n = 2 * T + 1, 2 * T - 1
    g = [list(r) for r in construct_binary_lanes(k)]
    for t in range(T):
        for r in range(2 * t + 1, m):
            g[r][2 * t] = "."                      # SAI: mở làn bất kể bit
    return ["".join(r) for r in g]

for k in (5, 8, 100):
    print(k, "-> sai:", count_paths(wrong_all_lanes(k)), " (kỳ vọng", k, ", 2^T-1 =", 2 ** k.bit_length() - 1, ")")

5 -> sai: 7  (kỳ vọng 5 , 2^T-1 = 7 )
8 -> sai: 15  (kỳ vọng 8 , 2^T-1 = 15 )
100 -> sai: 127  (kỳ vọng 100 , 2^T-1 = 127 )


**Cách đúng:** `if k >> t & 1:` mới mở làn `t`.

### Lỗi 4 — Đặt đích ở sai chỗ (đích không phải ô cuối hàng cuối)

`count_paths` luôn tính tới ô `(m-1, n-1)`. Nếu lưới dựng thừa hàng/cột (ví dụ `n = 2T` thay vì `2T-1`), ô đích bị lệch sang cột không nằm trên hàng cuối đã nối: đếm có thể là `0`.

In [19]:
def wrong_extra_column(k):
    g = [r + "#" for r in construct_binary_lanes(k)]    # SAI: thêm một cột chặn bên phải -> đích bị chặn
    return g

for k in (1, 6, 11):
    g = wrong_extra_column(k)
    print(k, "đích =", repr(g[-1][-1]), "đếm =", count_paths(g), "(kỳ vọng", k, ")")

1

 đích = '#' đếm = 0 (kỳ vọng 1 )
6 đích = '#' đếm = 0 (kỳ vọng 6 )
11 đích = '#' đếm = 0 (kỳ vọng 11 )


**Cách đúng:** `n = 2T − 1` và đích `(m−1, n−1)` chính là đáy của làn cao nhất.

## 14. Mẹo & nhận dạng pattern

- **"Dựng cấu trúc có đúng `k` đối tượng"** + `k` nhỏ so với kích thước tối đa → nghĩ tới **biểu diễn nhị phân** và các "bộ nhân đôi".
- **Đếm đường đi trên lưới** là `f = trên + trái`; thiết kế các ô để `f` chạy theo ý muốn.
- **Hình thoi `2×2`** = nhân đôi; **nối tiếp** hai lưới = nhân; **làn song song riêng biệt** = cộng.
- Muốn **cộng** các nhánh, hãy tách nhánh bằng ô chặn để chúng không "chảy" vào nhau.
- **Bài nhiều đáp án**: test bằng hàm kiểm tra (checker): đếm đường đi, kích thước, ký tự hợp lệ; đừng so khớp chuỗi.
- Kiểm chứng toàn bộ miền nhỏ của đề (`k ≤ 1000`) thay vì tin lập luận tay.
- Với bài Premium không có code mẫu, hãy **tự viết checker** từ định nghĩa đề trước khi viết bộ dựng.

## 15. Biến thể

1. **Giới hạn lớn hơn** (`k` tới `10^6` trong lưới `25×25`?): cách này cần `T = 20` ⇒ `41×39`; vượt `25`. Cần cách dựng chặt hơn (ví dụ tích các gạch nhỏ hoặc biểu diễn theo cơ số lớn hơn). Ta kiểm tra sức chứa của cách hiện tại.
2. **Tìm lưới nhỏ nhất theo diện tích** cho một `k`: thử tất cả cách nối tiếp theo ước + cách nhị phân (cách 3), hoặc tìm kiếm.
3. **Có thể đi cả lên/trái?** bài khác hẳn (đồ thị có chu trình, số đường đi vô hạn).
4. **Cho trước lưới, hỏi có bao nhiêu đường:** chính là `count_paths` (mô-đun hoá nếu lớn).

Ta tính `k` lớn nhất mà cách 2 dựng được trong lưới `25×25`, và kiểm chứng ranh giới.

In [20]:
def max_k_for_limit(limit=25):
    T = 0
    while 2 * (T + 1) + 1 <= limit:          # m = 2T+1 <= limit
        T += 1
    return T, (1 << T) - 1

T, kmax = max_k_for_limit(25)
print("T tối đa =", T, "=> k lớn nhất dựng được:", kmax)
g = Solution().constructGrid(kmax)
print("k =", kmax, "-> lưới", len(g), "x", len(g[0]), "đếm =", count_paths(g))
assert count_paths(g) == kmax and len(g) <= 25 and len(g[0]) <= 25
g = Solution().constructGrid(kmax + 1)
print("k =", kmax + 1, "-> lưới", len(g), "x", len(g[0]), "(vượt 25 hàng:", len(g) > 25, ")")

T tối đa = 12 => k lớn nhất dựng được: 4095
k = 4095 -> lưới 25 x 23 đếm = 4095
k = 4096 -> lưới 27 x 25 (vượt 25 hàng: True )


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Liệt kê lưới nhỏ | thử mọi lưới ≤ 16 ô, giữ lưới nhỏ nhất | `O(2^{mn}·mn)` | `O(mn)` | trọng tài và khám phá `k` nhỏ |
| 2. Hình thoi + làn nhị phân (nộp) | `f(S_t)=2^t`, làn theo bit | `O(T²)` | `O(T²)` | luôn dùng cho đề; đơn giản, chắc chắn |
| 3. Nối tiếp theo ước | nhân hai lưới, chọn lưới nhỏ hơn | `O(d(k)·T²)` | `O(T²)` | khi cần lưới nhỏ hơn |

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** `count_paths(["...", "...", "..."])` và `count_paths(["..", "#."])` bằng bao nhiêu?

**Bài 2 (tính tay).** Với lưới `["...", "#..", "#.."]`, viết bảng `f` từng ô và chỉ ra `f` tại đích.

**Bài 3 (điền TODO).** Hoàn thiện `count_paths_todo(grid)` bằng DP.

**Bài 4 (tự viết lại).** Viết `construct_power_of_two(T)` trả lưới có đúng `2^T` đường đi bằng chuỗi hình thoi (kích thước `(T+1)×(T+1)`).

**Bài 5 (biến thể / mini challenge).** Viết `construct_product(a, b)` trả lưới có đúng `a·b` đường đi bằng phép nối tiếp hai lưới dựng bởi `Solution().constructGrid`, rồi kiểm tra bằng `count_paths`. Nó có hợp lệ (≤ 25) với mọi `a·b ≤ 1000` không? Nếu không, trả mảng rỗng.

In [21]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def count_paths_todo(grid):
    m, n = len(grid), len(grid[0])
    # TODO 1: nếu ô đầu hoặc ô cuối là '#', trả 0
    # TODO 2: f[0][0] = 1; mỗi ô '.', f[i][j] = f[i-1][j] + f[i][j-1] (thiếu ô thì 0); ô '#' -> 0
    # TODO 3: trả f[m-1][n-1]
    pass

# Bài 4 — TODO
def construct_power_of_two(T):
    # TODO: lưới (T+1)x(T+1), mở ô (i, j) khi abs(i - j) <= 1
    pass

# Bài 5 — TODO
def construct_product(a, b):
    # TODO: series(Solution().constructGrid(a), Solution().constructGrid(b)); kiểm tra kích thước <= 25 else []
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: lưới trống `3×3` là số tổ hợp.
- Mức 2: `C(4, 2) = 6`. Lưới `["..", "#."]` chỉ có đường `(0,0)→(0,1)→(1,1)`.
- Mức 3: `6` và `1`.

**Bài 2.**
- Mức 1: `f(0,0)=1`, hàng đầu toàn `1`.
- Mức 2: cột 0 từ hàng 1 là `#` nên `f=0`; `f(1,1) = f(0,1) + f(1,0) = 1 + 0`.
- Mức 3: bảng `[[1,1,1],[0,1,2],[0,1,3]]`, đích `3`.

**Bài 3.**
- Mức 1: hai vòng for, xử lý ô `(0,0)` riêng.
- Mức 2: kiểm tra biên `i > 0`, `j > 0`.
- Mức 3: copy `count_paths` (hoặc tự viết lại không nhìn).

**Bài 4.**
- Mức 1: dải `|i-j| ≤ 1` đã gặp ở mục 4.
- Mức 2: `n = T + 1`.
- Mức 3: `["".join("." if abs(i - j) <= 1 else "#" for j in range(n)) for i in range(n)]`.

**Bài 5.**
- Mức 1: `series(A, B)` nhân số đường đi.
- Mức 2: kích thước cộng: `(m1 + m2 - 1) × (n1 + n2 - 1)`.
- Mức 3: kiểm tra `len(g) <= 25 and len(g[0]) <= 25` rồi mới trả.

## 19. Đáp án tham khảo

In [22]:
# Bài 1
assert count_paths(["...", "...", "..."]) == 6
assert count_paths(["..", "#."]) == 1

# Bài 2
assert f_table(["...", "#..", "#.."]) == [[1, 1, 1], [0, 1, 2], [0, 1, 3]]

# Bài 3
def count_paths_todo_ans(grid):
    m, n = len(grid), len(grid[0])
    if grid[0][0] == "#" or grid[m - 1][n - 1] == "#":
        return 0
    f = [[0] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            if grid[i][j] == "#":
                continue
            if i == 0 and j == 0:
                f[i][j] = 1
            else:
                f[i][j] = (f[i - 1][j] if i > 0 else 0) + (f[i][j - 1] if j > 0 else 0)
    return f[m - 1][n - 1]

for _ in range(300):
    m, n = rng.randint(1, 6), rng.randint(1, 6)
    g = ["".join(rng.choice(".#.") for _ in range(n)) for _ in range(m)]
    assert count_paths_todo_ans(g) == count_paths(g)

# Bài 4
def construct_power_of_two_ans(T):
    n = T + 1
    return ["".join("." if abs(i - j) <= 1 else "#" for j in range(n)) for i in range(n)]

for T in range(0, 11):
    g = construct_power_of_two_ans(T)
    assert count_paths(g) == 2 ** T, T
print("Bài 1–4 OK")

Bài 1–4 OK


In [23]:
# Bài 5
def construct_product_ans(a, b):
    g = series(Solution().constructGrid(a), Solution().constructGrid(b))
    if len(g) > 25 or len(g[0]) > 25:
        return []
    return g

assert count_paths(construct_product_ans(3, 5)) == 15
ok = bad = 0
for a in range(1, 40):
    for b in range(1, 40):
        if a * b > 1000:
            continue
        g = construct_product_ans(a, b)
        if g:
            assert count_paths(g) == a * b
            ok += 1
        else:
            bad += 1
print("a*b ≤ 1000, a,b < 40: dựng được", ok, "cặp; vượt 25:", bad, "cặp (thay bằng Solution().constructGrid(a*b))")

a*b ≤ 1000, a,b < 40: dựng được 1407 cặp; vượt 25: 0 cặp (thay bằng Solution().constructGrid(a*b))


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "xương sống nhân đôi + làn theo bit":**

```python
T = k.bit_length()
m, n = 2 * T + 1, 2 * T - 1
g = [["#"] * n for _ in range(m)]
for t in range(T):
    g[2*t][2*t] = g[2*t+1][2*t] = "."               # S_t và góc dưới-trái
    if 2*t + 1 < n:
        g[2*t][2*t+1] = g[2*t+1][2*t+1] = "."       # hình thoi
    if t < T - 1:
        g[2*t+1][2*t+2] = "."                       # nối sang tầng sau
    if k >> t & 1:
        for r in range(2*t + 1, m):
            g[r][2*t] = "."                         # làn t
for j in range(n):
    g[m-1][j] = "."                                 # hàng cuối gom về đích
return ["".join(r) for r in g]
```

**Checklist trước khi submit:**

- [ ] Hai ô góc `(0,0)` và `(m-1,n-1)` đều là `'.'`.
- [ ] `m = 2T+1 ≤ 25`, `n = 2T-1 ≤ 25` (với `k ≤ 1000` thì `T ≤ 10`).
- [ ] Hai làn cách nhau một cột `'#'`; chỉ mở làn của bit bật.
- [ ] Không dùng `[['#']*n]*m` (cùng tham chiếu).
- [ ] Đã kiểm chứng `count_paths(constructGrid(k)) == k` cho mọi `k = 1..1000`.
- [ ] Đã đối chiếu tên phương thức và kiểu trả về (`List[str]`) nếu có tài khoản Premium.